# MM26 Data Pipeline Explorer

This notebook is a guided tour of the entire data pipeline — from raw source files through to the model-ready gold layer. Each section explains **what the data is**, **where it comes from**, and **what transformations were applied** to get it.

---

## Pipeline Overview

```
Kaggle CSVs  ──┐
               ├──► Bronze (raw parquet snapshots)
CBBD API     ──┘         │
                         ▼
                    Silver (cleaned, joined, one row per team-game)
                         │  ├── ELO ratings (MOV + carry-over + home court)
                         │  ├── Heat scores (actual vs expected margin)
                         │  └── Consensus betting lines
                         ▼
                    Gold (24 pairwise features for ML)
                         │  ├── XGBoost classifier (isotonically calibrated)
                         │  ├── Dynamic probability clipping (seed-aware)
                         │  └── Monte Carlo bracket simulation (100K)
                         ▼
                    submission.csv
```

| Layer | Files | Description |
|---|---|---|
| **Bronze/Kaggle** | `bronze/kaggle/*.parquet` | Kaggle CSVs frozen as parquet snapshots |
| **Bronze/CBBD** | `bronze/cbbd/*.parquet` | Live API data: games, box scores, lines |
| **Silver** | `silver/*.parquet` | Cleaned and joined; canonical team-game grain |
| **Gold** | `gold/*.parquet` | Aggregated per-team-season features; pairwise matchup features |

All files are read from `artifacts/latest/` which always points to the most recent pipeline run.

In [1]:
from __future__ import annotations
import json
from pathlib import Path
import polars as pl
import polars.selectors as cs
import pyarrow as pa

ROOT = Path("..").resolve()
BASE = ROOT / "artifacts" / "latest"

# Load the run manifest — the pipeline writes this at completion and records
# every decision it made: mode, target season, CBBD windows, row counts, etc.
manifest = json.loads((BASE / "run_manifest.json").read_text(encoding="utf-8"))

print(f"Run ID        : {manifest['run_id']}")
print(f"Mode          : {manifest['mode']}")
print(f"Target season : {manifest['target_season']}")
print(f"CBBD seasons  : {manifest['ingest']['cbbd']['historical_seasons']}")
print()
print("CBBD dataset row counts:")
for name, info in manifest["ingest"]["cbbd"]["datasets"].items():
    print(f"  {name:15s}  status={info['status']:7s}  rows={info['rows']:,}")

Run ID        : 20260318T021522Z_3f1b55d8
Mode          : manual
Target season : 2026
CBBD seasons  : [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

CBBD dataset row counts:
  games            status=ok       rows=60,360
  game_teams       status=ok       rows=114,706
  lines            status=ok       rows=124,880


---
## Part 1 — Bronze Layer: Kaggle Source Data

The Kaggle dataset provides the authoritative game results and team registry for both the men's and women's tournaments. The pipeline copies every CSV verbatim into parquet snapshots under `bronze/kaggle/` so they're immutable and fast to query.

**Key Kaggle files used downstream:**
- `MTeams` / `WTeams` — team IDs and names (the master registry)
- `MRegularSeasonDetailedResults` — men's regular season box scores from 2003 onward
- `WRegularSeasonDetailedResults` — women's regular season box scores from 1998 onward
- `MNCAATourneyCompactResults` / `WNCAATourneyCompactResults` — historical tournament outcomes
- `SampleSubmissionStage2` — defines the exact set of matchups we must predict

In [2]:
# --- Bronze / Kaggle: Team registry ---
# MTeams is the authoritative list of men's Division-I programs.
# TeamID is the key that links every other dataset.
# FirstD1Season / LastD1Season tell us when a school was D1 (LastD1Season=2026 means currently active).
m_teams = pl.read_parquet(BASE / "bronze/kaggle/MTeams.parquet")
print(f"Men's teams: {m_teams.height} rows")
print(m_teams.head(10))

Men's teams: 381 rows
shape: (10, 4)
┌────────┬───────────────┬───────────────┬──────────────┐
│ TeamID ┆ TeamName      ┆ FirstD1Season ┆ LastD1Season │
│ ---    ┆ ---           ┆ ---           ┆ ---          │
│ i64    ┆ str           ┆ i64           ┆ i64          │
╞════════╪═══════════════╪═══════════════╪══════════════╡
│ 1101   ┆ Abilene Chr   ┆ 2014          ┆ 2026         │
│ 1102   ┆ Air Force     ┆ 1985          ┆ 2026         │
│ 1103   ┆ Akron         ┆ 1985          ┆ 2026         │
│ 1104   ┆ Alabama       ┆ 1985          ┆ 2026         │
│ 1105   ┆ Alabama A&M   ┆ 2000          ┆ 2026         │
│ 1106   ┆ Alabama St    ┆ 1985          ┆ 2026         │
│ 1107   ┆ SUNY Albany   ┆ 2000          ┆ 2026         │
│ 1108   ┆ Alcorn St     ┆ 1985          ┆ 2026         │
│ 1109   ┆ Alliant Intl  ┆ 1985          ┆ 1991         │
│ 1110   ┆ American Univ ┆ 1985          ┆ 2026         │
└────────┴───────────────┴───────────────┴──────────────┘


In [3]:
# --- Bronze / Kaggle: Regular season detailed results ---
# This is the richest game-level dataset Kaggle provides.
# Each row is one game with full box-score totals (field goals, rebounds, etc.)
# The winner is always in the W* columns; loser in L* columns.
# DayNum is days since "Day 0" for that season — use MSeasons.DayZero to get calendar dates.
m_reg = pl.read_parquet(BASE / "bronze/kaggle/MRegularSeasonDetailedResults.parquet")
w_reg = pl.read_parquet(BASE / "bronze/kaggle/WRegularSeasonDetailedResults.parquet")
print(f"Men's detailed results: {m_reg.height:,} rows | Seasons: {m_reg['Season'].min()}–{m_reg['Season'].max()}")
print(f"Women's detailed results: {w_reg.height:,} rows | Seasons: {w_reg['Season'].min()}–{w_reg['Season'].max()}")
print()
print("Columns:", m_reg.columns)
m_reg.head(5)

Men's detailed results: 124,529 rows | Seasons: 2003–2026
Women's detailed results: 87,187 rows | Seasons: 2010–2026

Columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']


Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
i64,i64,i64,i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
2003,10,1104,68,1328,62,"""N""",0,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20
2003,10,1272,70,1393,63,"""N""",0,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16
2003,11,1266,73,1437,61,"""N""",0,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23
2003,11,1296,56,1457,50,"""N""",0,18,38,3,9,17,31,6,19,11,12,14,2,18,18,49,6,22,8,15,17,20,9,19,4,3,23
2003,11,1400,77,1208,71,"""N""",0,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14


In [4]:
# --- Bronze / Kaggle: NCAA Tournament results ---
# Compact results have only final scores. These are the ground-truth labels
# for every past tournament game and also define what we're predicting.
m_tourney = pl.read_parquet(BASE / "bronze/kaggle/MNCAATourneyCompactResults.parquet")
m_seeds   = pl.read_parquet(BASE / "bronze/kaggle/MNCAATourneySeeds.parquet")
sample    = pl.read_parquet(BASE / "bronze/kaggle/SampleSubmissionStage2.parquet")

print(f"Tournament results (M): {m_tourney.height:,} rows | Seasons: {m_tourney['Season'].min()}–{m_tourney['Season'].max()}")
print(f"Tournament seeds (M):   {m_seeds.height:,} rows")
print(f"Stage 2 submission rows to predict: {sample.height:,}")
print()
print("Sample submission format (ID = Season_TeamLow_TeamHigh, Pred = P(TeamLow wins))")
sample.head(5)

Tournament results (M): 2,585 rows | Seasons: 1985–2025
Tournament seeds (M):   2,694 rows
Stage 2 submission rows to predict: 132,133

Sample submission format (ID = Season_TeamLow_TeamHigh, Pred = P(TeamLow wins))


ID,Pred
str,f64
"""2026_1101_1102""",0.5
"""2026_1101_1103""",0.5
"""2026_1101_1104""",0.5
"""2026_1101_1105""",0.5
"""2026_1101_1106""",0.5


---
## Part 2 — Bronze Layer: CBBD API Data

The College Basketball Data API (CBBD) enriches the pipeline with three datasets that Kaggle doesn't provide:

| Dataset | Description | Grain |
|---|---|---|
| `games.parquet` | Game metadata: final scores, Elo ratings, seeds, excitement index | 1 row per game |
| `game_teams.parquet` | Team-level box scores per game (the "four factors" and more) | 2 rows per game (one per team) |
| `lines.parquet` | Raw betting lines per provider per game | 1 row per game × provider |

The pipeline fetches the **last 10 seasons (2016–2025)** using a **35-day sliding window** per season so it never hits the API's 3,000-result-per-call cap.

In [5]:
# --- Bronze / CBBD: Games ---
# One row per game. Includes final scores, pre/post Elo for both teams,
# tournament seeds, site (neutral/home), conference flag, and an excitment index.
# `home_score` / `away_score` are the actual final points scored.
cbbd_games = pl.read_parquet(BASE / "bronze/cbbd/games.parquet")
print(f"CBBD games: {cbbd_games.height:,} rows  |  {cbbd_games['season'].min()}–{cbbd_games['season'].max()}")
print("Columns:", cbbd_games.columns)
cbbd_games.head(5)

CBBD games: 60,360 rows  |  2016–2025
Columns: ['game_id', 'season', 'season_type', 'status', 'start_date', 'home_team_id', 'home_team', 'home_conference', 'home_score', 'away_team_id', 'away_team', 'away_conference', 'away_score', 'neutral_site', 'conference_game', 'notes', 'home_elo_start', 'home_elo_end', 'away_elo_start', 'away_elo_end', 'home_seed', 'away_seed', 'excitement']


game_id,season,season_type,status,start_date,home_team_id,home_team,home_conference,home_score,away_team_id,away_team,away_conference,away_score,neutral_site,conference_game,notes,home_elo_start,home_elo_end,away_elo_start,away_elo_end,home_seed,away_seed,excitement
i64,i64,str,str,str,i64,str,str,f64,i64,str,str,f64,bool,bool,str,f64,f64,f64,f64,f64,f64,f64
57155,2016,"""regular""","""final""","""2016-01-16 21:00:00+00:00""",328,"""UTSA""","""CUSA""",71.0,327,"""UTEP""","""CUSA""",67.0,false,true,null,1298.0,1319.0,1633.0,1612.0,null,null,5.6
13514,2023,"""regular""","""final""","""2022-11-30 00:00:00+00:00""",181,"""Morehead State""","""OVC""",109.0,590,"""Kentucky Christian""",null,62.0,false,false,null,1643.0,1806.0,1448.0,1285.0,null,null,1.6
28594,2021,"""regular""","""final""","""2021-02-12 23:00:00+00:00""",142,"""Liberty""","""ASUN""",73.0,203,"""North Florida""","""ASUN""",61.0,false,true,null,1912.0,1897.0,1578.0,1593.0,null,null,1.4
45100,2018,"""regular""","""final""","""2018-01-12 00:00:00+00:00""",216,"""Ohio State""","""Big Ten""",91.0,160,"""Maryland""","""Big Ten""",69.0,false,true,null,2038.0,2078.0,1951.0,1911.0,null,null,2.2
48765,2017,"""regular""","""final""","""2016-11-23 00:30:00+00:00""",8,"""American University""","""Patriot""",65.0,341,"""Wagner""","""NEC""",73.0,false,false,null,1472.0,1436.0,1540.0,1576.0,null,null,5.0


In [6]:
# --- Bronze / CBBD: Game Teams (box scores) ---
# Two rows per game — one per team. Contains the "four factors" and other advanced
# stats computed by CBBD: pace, offensive/defensive rating, field goal splits,
# rebounds, turnovers, free throw rate, etc.
# teamStats and opponentStats are flattened into columns like team_stats_points_total,
# team_stats_field_goals_made, team_stats_four_factors_effective_fg_pct, etc.
cbbd_gt = pl.read_parquet(BASE / "bronze/cbbd/game_teams.parquet")
print(f"CBBD game_teams: {cbbd_gt.height:,} rows  (2 rows per game = {cbbd_gt.height // 2:,} unique games)")
print(f"Columns ({len(cbbd_gt.columns)}):", cbbd_gt.columns[:16], "...")
cbbd_gt.select(["game_id", "season", "team", "opponent", "is_home", "pace",
                "game_type", "game_minutes"]).head(6)

CBBD game_teams: 114,706 rows  (2 rows per game = 57,353 unique games)
Columns (16): ['game_id', 'season', 'season_type', 'start_date', 'team_id', 'team', 'conference', 'opponent_id', 'opponent', 'opponent_conference', 'neutral_site', 'is_home', 'conference_game', 'game_type', 'game_minutes', 'pace'] ...


game_id,season,team,opponent,is_home,pace,game_type,game_minutes
i64,i64,str,str,bool,f64,str,f64
11928,2024,"""Ohio""","""Western Michigan""",true,69.5,"""TRNMNT""",40.0
39729,2019,"""Bethune-Cookman""","""Coppin State""",false,null,"""STD""",40.0
6015,2024,"""Robert Morris""","""Xavier""",false,72.5,"""STD""",40.0
44902,2018,"""Alcorn State""","""Jackson State""",true,60.5,"""STD""",40.0
33929,2020,"""Fairleigh Dickinson""","""Wagner""",false,65.0,"""STD""",40.0
20634,2022,"""Alabama""","""Davidson""",true,67.0,"""STD""",40.0


In [7]:
# --- Bronze / CBBD: Lines ---
# One row per game × betting provider (e.g. "draftkings", "numberfire", "consensus").
# spread: negative = home team favoured (e.g. -3.5 means home favoured by 3.5 pts)
# spread_open: the opening line before market movement
# over_under: total projected points scored
# home_moneyline / away_moneyline: money line odds for each side
cbbd_lines = pl.read_parquet(BASE / "bronze/cbbd/lines.parquet")
print(f"CBBD lines: {cbbd_lines.height:,} rows")
print(f"Providers: {sorted(cbbd_lines['provider'].drop_nulls().unique().to_list())}")
print()
cbbd_lines.select(["game_id", "season", "home_team", "away_team",
                   "provider", "spread", "over_under", "home_moneyline"]).head(8)

CBBD lines: 124,880 rows
Providers: ['ESPN BET', 'consensus', 'numberfire', 'teamrankings']



game_id,season,home_team,away_team,provider,spread,over_under,home_moneyline
i64,i64,str,str,str,f64,f64,f64
3776,2025,"""Illinois""","""Maryland""","""ESPN BET""",-5.5,156.5,-240.0
48272,2017,"""Texas""","""UL Monroe""","""numberfire""",-17.5,148.0,null
52870,2017,"""La Salle""","""Saint Joseph's""","""numberfire""",-6.0,145.0,-255.0
59734,2016,"""Hampton""","""Savannah State""","""numberfire""",-6.5,127.0,-285.0
58851,2016,"""Northern Iowa""","""Illinois State""","""teamrankings""",-5.5,129.0,-230.0
36410,2019,"""Idaho""","""Nicholls""","""numberfire""",-5.5,147.0,null
39405,2019,"""Alcorn State""","""Arkansas-Pine Bluff""","""teamrankings""",4.0,125.0,null
25047,2021,"""UT Arlington""","""Northwestern State""","""numberfire""",-7.5,152.5,-370.0


In [8]:

# --- Lines quality check: verify lines match expected games ---
# 1. Every lines row should reference a game_id that exists in games.parquet.
#    Orphaned lines (no matching game) indicate a data integrity issue.
# 2. Lines are at grain: game × provider. Dividing by unique providers gives
#    an expected upper bound on unique games with lines.
# 3. Coverage = what % of games in the games table have at least one line.

all_game_ids   = set(cbbd_games["game_id"].to_list())
line_game_ids  = set(cbbd_lines["game_id"].drop_nulls().to_list())

games_with_lines = line_game_ids & all_game_ids          # matched
orphaned_lines   = line_game_ids - all_game_ids          # lines with no game record

total_lines                = cbbd_lines.height
unique_games_in_lines      = cbbd_lines["game_id"].drop_nulls().n_unique()
unique_providers_in_lines  = cbbd_lines["provider"].drop_nulls().n_unique()
lines_coverage_pct         = 100 * len(games_with_lines) / len(all_game_ids)

print("=== Lines ↔ Games integrity check ===")
print(f"Total line rows             : {total_lines:,}")
print(f"Unique game IDs in lines    : {unique_games_in_lines:,}")
print(f"Unique providers            : {unique_providers_in_lines}")
print(f"Total games in games table  : {len(all_game_ids):,}")
print(f"Games WITH at least one line: {len(games_with_lines):,}  ({lines_coverage_pct:.1f}% of all games)")
print(f"Games WITHOUT any line      : {len(all_game_ids) - len(games_with_lines):,}")
print(f"Orphaned lines (no game)    : {len(orphaned_lines):,}", end="")
print("  ✓ OK" if len(orphaned_lines) == 0 else "  ⚠ UNEXPECTED — investigate these game IDs")

print()
# Expected lines per game ≃ unique_games_in_lines × avg providers per game
avg_providers = cbbd_lines.group_by("game_id").agg(
    pl.col("provider").drop_nulls().n_unique().alias("provider_count")
)["provider_count"].mean()
print(f"Avg providers per game      : {avg_providers:.2f}")
print(f"Expected total rows (≈)     : {unique_games_in_lines:,} games × {avg_providers:.2f} providers ≈ {int(unique_games_in_lines * avg_providers):,}  (actual: {total_lines:,})")

print()
# Null spread / over_under rates — high nulls = providers that only supply partial data
null_spread = cbbd_lines["spread"].is_null().sum()
null_ou     = cbbd_lines["over_under"].is_null().sum()
print(f"Null spread values          : {null_spread:,} / {total_lines:,}  ({100*null_spread/total_lines:.1f}%)")
print(f"Null over_under values      : {null_ou:,}  / {total_lines:,}  ({100*null_ou/total_lines:.1f}%)")


=== Lines ↔ Games integrity check ===
Total line rows             : 124,880
Unique game IDs in lines    : 60,360
Unique providers            : 4
Total games in games table  : 60,360
Games WITH at least one line: 60,360  (100.0% of all games)
Games WITHOUT any line      : 0
Orphaned lines (no game)    : 0  ✓ OK

Avg providers per game      : 1.97
Expected total rows (≈)     : 60,360 games × 1.97 providers ≈ 119,088  (actual: 124,880)

Null spread values          : 11,898 / 124,880  (9.5%)
Null over_under values      : 18,509  / 124,880  (14.8%)


---
## Part 3 — Silver Layer: Cleaned & Joined Data

The silver layer transforms the raw bronze data into a clean, analysis-ready format.

### What happens in the transform step:

1. **`game_fact.parquet`** — Kaggle detailed results are "exploded" from winner/loser format into one row per team per game (so each game becomes 2 rows). This makes it easy to compute per-team season stats.

2. **`team_dim.parquet`** — A dimension table of all teams (men's and women's) with their metadata.

3. **`team_id_map.parquet`** — CBBD uses its own team name strings; this table maps CBBD team names → Kaggle TeamIDs by fuzzy name matching. Coverage report shows how many were matched.

4. **`cbbd_games.parquet`** — CBBD games enriched with mapped Kaggle TeamIDs and a canonical `team_low / team_high` pair key.

5. **`cbbd_game_fact.parquet`** — The CBBD box score data (from `game_teams`), one row per team per game.

6. **`cbbd_line_consensus.parquet`** — Provider lines aggregated to a single consensus spread per game using the median across non-null providers.

7. **`cbbd_lines.parquet`** — All raw provider lines with Kaggle TeamIDs mapped in.

8. **`quality_flags.parquet`** — A pass/fail quality flag per row indicating whether a game has all required features for model training.

In [9]:
# --- Silver: game_fact ---
# One row per team per game. The Kaggle detailed results are in winner/loser format.
# The pipeline "flips" them so every team appears on both sides of each game.
# sex = 'M' or 'W', team_loc = 'H'(ome) / 'A'(way) / 'N'(eutral)
# win = 1 if this team won, 0 if they lost
# game_key = f"{season}_{team_low}_{team_high}" — a stable game identifier
game_fact = pl.read_parquet(BASE / "silver/game_fact.parquet")
print(f"game_fact: {game_fact.height:,} rows — {game_fact.height // 2:,} unique games")
print(f"Seasons: {game_fact['season'].min()}–{game_fact['season'].max()}")
print()
game_fact.head(6)

game_fact: 423,432 rows — 211,716 unique games
Seasons: 2003–2026



sex,season,day_num,team_id,opp_team_id,team_score,opp_score,team_loc,num_ot,win,team_low,team_high,game_key
str,i64,i64,i64,i64,f64,f64,str,i64,i32,i64,i64,str
"""M""",2003,10,1104,1328,68.0,62.0,"""N""",0,1,1104,1328,"""M_2003_10_1104_1328"""
"""M""",2003,10,1272,1393,70.0,63.0,"""N""",0,1,1272,1393,"""M_2003_10_1272_1393"""
"""M""",2003,11,1266,1437,73.0,61.0,"""N""",0,1,1266,1437,"""M_2003_11_1266_1437"""
"""M""",2003,11,1296,1457,56.0,50.0,"""N""",0,1,1296,1457,"""M_2003_11_1296_1457"""
"""M""",2003,11,1400,1208,77.0,71.0,"""N""",0,1,1208,1400,"""M_2003_11_1208_1400"""
"""M""",2003,11,1458,1186,81.0,55.0,"""H""",0,1,1186,1458,"""M_2003_11_1186_1458"""


In [10]:
# --- Silver: team_dim ---
# Dimension table for all teams. Links sex + team_id to a team name.
# first_d1_season / last_d1_season indicate D1 membership windows.
team_dim = pl.read_parquet(BASE / "silver/team_dim.parquet")
print(f"team_dim: {team_dim.height} teams  (M: {team_dim.filter(pl.col('sex')=='M').height}, W: {team_dim.filter(pl.col('sex')=='W').height})")
team_dim.head(8)

team_dim: 760 teams  (M: 381, W: 379)


sex,team_id,team_name,first_d1_season,last_d1_season
str,i64,str,i64,i64
"""M""",1101,"""Abilene Chr""",2014,2026
"""M""",1102,"""Air Force""",1985,2026
"""M""",1103,"""Akron""",1985,2026
"""M""",1104,"""Alabama""",1985,2026
"""M""",1105,"""Alabama A&M""",2000,2026
"""M""",1106,"""Alabama St""",1985,2026
"""M""",1107,"""SUNY Albany""",2000,2026
"""M""",1108,"""Alcorn St""",1985,2026


In [11]:
# --- Silver: team_id_map (CBBD → Kaggle) ---
# The CBBD API uses its own team names; Kaggle uses numeric TeamIDs.
# The pipeline normalises both sides (lowercase, remove punctuation, expand "&" → "and")
# and performs exact string matching, then fuzzy matching for near-misses.
# mapped=True means a Kaggle TeamID was found; mapped=False = unmatched.
team_id_map = pl.read_parquet(BASE / "silver/team_id_map.parquet")
matched   = team_id_map.filter(pl.col("mapped") == True)
unmatched = team_id_map.filter(pl.col("mapped") == False)
print(f"Mapping coverage: {matched.height} / {team_id_map.height} matched  ({100*matched.height/team_id_map.height:.1f}%)")
print(f"Unmatched CBBD team names ({unmatched.height}):")
print(unmatched["cbbd_team_name"].to_list()[:20])

Mapping coverage: 366 / 1209 matched  (30.3%)
Unmatched CBBD team names (843):
['Pfeiffer', 'Washington and Lee', 'Wilmington (OH)', 'Plattsburgh St', 'Maine-Fort Kent', 'Peru State', 'York (NY)', 'Calumet College', 'East Texas Baptist', 'Linfield College', 'SUNY-Canton', 'Crown College', 'SUNY-Oneonta', 'Washington College (MD)', 'Humboldt State', 'Lincoln (PA)', 'Clarks Summit', 'Texas Wesleyan', 'Rhodes', 'Southern-New Orleans']


In [12]:
# --- Silver: cbbd_line_consensus ---
# Raw lines from multiple providers are aggregated to a single consensus per game.
# consensus_home_spread = median of all non-null provider spreads
# provider_count = how many providers contributed data for this game
# Games with only 1 provider are less reliable than games with 3+.
cbbd_consensus = pl.read_parquet(BASE / "silver/cbbd_line_consensus.parquet")
print(f"cbbd_line_consensus: {cbbd_consensus.height:,} rows")
print("Columns:", cbbd_consensus.columns)
print()
# Provider coverage distribution
provider_dist = cbbd_consensus.group_by("provider_count").agg(pl.len().alias("games")).sort("provider_count")
print("Provider count distribution:")
print(provider_dist)
cbbd_consensus.head(5)

cbbd_line_consensus: 60,360 rows
Columns: ['game_id', 'season', 'start_date', 'home_team_id', 'away_team_id', 'consensus_home_spread', 'consensus_home_spread_open', 'consensus_over_under', 'provider_count', 'kaggle_home_team_id', 'kaggle_away_team_id', 'team_low', 'team_high', 'consensus_low_spread']

Provider count distribution:
shape: (4, 2)
┌────────────────┬───────┐
│ provider_count ┆ games │
│ ---            ┆ ---   │
│ u32            ┆ u32   │
╞════════════════╪═══════╡
│ 0              ┆ 5792  │
│ 1              ┆ 16866 │
│ 2              ┆ 10884 │
│ 3              ┆ 26818 │
└────────────────┴───────┘


game_id,season,start_date,home_team_id,away_team_id,consensus_home_spread,consensus_home_spread_open,consensus_over_under,provider_count,kaggle_home_team_id,kaggle_away_team_id,team_low,team_high,consensus_low_spread
i64,i64,str,i64,i64,f64,f64,f64,u32,i64,i64,i64,i64,f64
53967,2016,"""2015-11-13 16:00:00+00:00""",58,825,null,null,null,0,1162,null,1162,1162,null
53968,2016,"""2015-11-13 16:00:00+00:00""",79,337,null,null,null,1,1185,1436,1185,1436,null
53969,2016,"""2015-11-13 19:00:00+00:00""",316,615,null,null,null,0,null,null,null,null,null
53970,2016,"""2015-11-13 20:00:00+00:00""",360,83,-7.5,null,125.0,2,1463,1193,1193,1463,7.5
53971,2016,"""2015-11-13 21:00:00+00:00""",201,1101,null,null,null,0,1315,null,1315,1315,null


In [13]:
# --- Silver: cbbd_game_fact (box score per team) ---
# One row per team per game pulled from the CBBD game_teams endpoint.
# Contains pace (possessions per 40 mins), plus nested teamStats/opponentStats
# flattened into columns: team_stats_field_goals_made, team_stats_four_factors_effective_fg_pct, etc.
cbbd_gf = pl.read_parquet(BASE / "silver/cbbd_game_fact.parquet")
print(f"cbbd_game_fact: {cbbd_gf.height:,} rows  ({cbbd_gf.height // 2:,} unique games)")
print(f"Columns ({len(cbbd_gf.columns)}): {cbbd_gf.columns}")
# Show a focused slice of the box-score stats
stat_cols = [c for c in cbbd_gf.columns if "team_stats" in c]
print(f"\nBox score stat columns ({len(stat_cols)}):")
print(stat_cols)

cbbd_game_fact: 114,706 rows  (57,353 unique games)
Columns (20): ['game_id', 'season', 'season_type', 'start_date', 'team_id', 'team', 'conference', 'opponent_id', 'opponent', 'opponent_conference', 'neutral_site', 'is_home', 'conference_game', 'game_type', 'game_minutes', 'pace', 'kaggle_team_id', 'kaggle_opp_team_id', 'sex', 'game_date']

Box score stat columns (0):
[]


In [14]:
# --- Silver: quality_flags ---
# Every row in game_fact gets a row_quality_pass flag.
# A row passes if all features needed for model training are non-null.
# The train/holdout split only uses rows where row_quality_pass = True.
quality = pl.read_parquet(BASE / "silver/quality_flags.parquet")
pass_rate = quality.filter(pl.col("row_quality_pass")).height / quality.height
print(f"Quality flags: {quality.height:,} total rows")
print(f"Pass rate: {pass_rate:.1%}  ({quality.filter(pl.col('row_quality_pass')).height:,} passing)")
print()
# Break down by sex and season
by_season = (
    quality
    .group_by(["sex", "season"])
    .agg(
        pl.len().alias("total_rows"),
        pl.col("row_quality_pass").sum().alias("passing"),
    )
    .with_columns((pl.col("passing") / pl.col("total_rows") * 100).round(1).alias("pass_pct"))
    .sort(["sex", "season"])
)
print(by_season)

Quality flags: 423,432 total rows
Pass rate: 100.0%  (423,432 passing)

shape: (41, 5)
┌─────┬────────┬────────────┬─────────┬──────────┐
│ sex ┆ season ┆ total_rows ┆ passing ┆ pass_pct │
│ --- ┆ ---    ┆ ---        ┆ ---     ┆ ---      │
│ str ┆ i64    ┆ u32        ┆ u32     ┆ f64      │
╞═════╪════════╪════════════╪═════════╪══════════╡
│ M   ┆ 2003   ┆ 9232       ┆ 9232    ┆ 100.0    │
│ M   ┆ 2004   ┆ 9142       ┆ 9142    ┆ 100.0    │
│ M   ┆ 2005   ┆ 9350       ┆ 9350    ┆ 100.0    │
│ M   ┆ 2006   ┆ 9514       ┆ 9514    ┆ 100.0    │
│ M   ┆ 2007   ┆ 10086      ┆ 10086   ┆ 100.0    │
│ …   ┆ …      ┆ …          ┆ …       ┆ …        │
│ W   ┆ 2022   ┆ 10120      ┆ 10120   ┆ 100.0    │
│ W   ┆ 2023   ┆ 10748      ┆ 10748   ┆ 100.0    │
│ W   ┆ 2024   ┆ 10828      ┆ 10828   ┆ 100.0    │
│ W   ┆ 2025   ┆ 10888      ┆ 10888   ┆ 100.0    │
│ W   ┆ 2026   ┆ 10958      ┆ 10958   ┆ 100.0    │
└─────┴────────┴────────────┴─────────┴──────────┘


---
## Part 4 — Gold Layer: Model-Ready Features

The gold layer aggregates game_fact into features suitable for a machine learning model. There are three tables:

| Table | Grain | Purpose |
|---|---|---|
| `team_season_features` | 1 row per team per season | Season-level averages: PPG, win rate, margin, ELO, heat, box scores, SOS |
| `game_features` | 1 row per game | Aggregated stats for each side of a historical game |
| `pairwise_features` | 1 row per matchup in submission | The actual ML input: 24 difference features for team_low vs team_high |

**The train/validate/predict split:**
- **Train**: seasons 2003–2024 (Kaggle detailed results)
- **Holdout/validation**: season 2025
- **Prediction target**: season 2026 (the current tournament)

In [15]:
# --- Gold: team_season_features ---
# One row per team per season. The pipeline computes these from the silver game_fact:
#   games_played, wins, losses
#   avg_pts_for, avg_pts_against, avg_margin (offensive/defensive efficiency proxies)
#   avg_num_ot (overtime rate — higher = more close games)
#   home_win_rate, away_win_rate (location-adjusted performance)
team_feats = pl.read_parquet(BASE / "gold/team_season_features.parquet")
print(f"team_season_features: {team_feats.height:,} rows")
print(f"Seasons: {team_feats['season'].min()}–{team_feats['season'].max()}")
print("Columns:", team_feats.columns)
print()
# Example: top 10 men's teams by average margin in 2025
(
    team_feats
    .filter((pl.col("season") == 2025) & (pl.col("sex") == "M"))
    .sort("avg_margin", descending=True)
    .join(team_dim.filter(pl.col("sex") == "M").select(["team_id", "team_name"]), on="team_id", how="left")
    .select(["team_name", "season", "games_played", "wins", "avg_pts_for", "avg_pts_against", "avg_margin"])
    .head(10)
)

team_season_features: 14,311 rows
Seasons: 2003–2026
Columns: ['sex', 'season', 'team_id', 'games_played', 'wins', 'losses', 'avg_pts_for', 'avg_pts_against', 'avg_margin', 'avg_num_ot', 'last5_avg_pts_for', 'last5_avg_pts_against', 'last5_avg_margin', 'last5_win_rate', 'win_rate']



team_name,season,games_played,wins,avg_pts_for,avg_pts_against,avg_margin
str,i64,u32,i32,f64,f64,f64
"""Duke""",2025,34,31,82.705882,61.911765,20.794118
"""Gonzaga""",2025,33,25,86.636364,69.636364,17.0
"""Florida""",2025,34,30,85.411765,69.235294,16.176471
"""Houston""",2025,34,30,74.205882,58.470588,15.735294
"""UC San Diego""",2025,32,28,77.9375,62.84375,15.09375
"""Maryland""",2025,33,25,81.666667,67.0,14.666667
"""Auburn""",2025,33,28,83.848485,69.606061,14.242424
"""VCU""",2025,33,27,76.333333,62.515152,13.818182
"""Texas Tech""",2025,33,25,80.909091,67.575758,13.333333


In [16]:
# --- Gold: pairwise_features ---
# This is the direct input to the model. One row per matchup in the submission file.
# Each row represents a potential tournament game: team_low (lower TeamID) vs team_high.
# The features are *differences* between the two teams' season stats, so positive values
# mean team_low has the advantage on that dimension.
# Feature columns that end in _diff = team_low_stat - team_high_stat.
# The target is: did team_low win? (1 = yes, 0 = no)
pairs = pl.read_parquet(BASE / "gold/pairwise_features.parquet")
print(f"pairwise_features: {pairs.height:,} rows  (= submission rows to predict)")
print("Columns:", pairs.columns)
print()
# Check how many have all features vs missing features
feature_cols = [c for c in pairs.columns if c not in ("ID", "season", "team_low", "team_high", "sex", "label")]
null_counts = pairs.select([pl.col(c).is_null().sum().alias(c) for c in feature_cols])
complete = pairs.filter(pl.all_horizontal([pl.col(c).is_not_null() for c in feature_cols])).height
print(f"Rows with all features populated: {complete:,} / {pairs.height:,}  ({100*complete/pairs.height:.1f}%)")
pairs.head(3)

pairwise_features: 132,133 rows  (= submission rows to predict)
Columns: ['ID', 'season', 'team_low', 'team_high', 'sex', 'games_low', 'wins_low', 'losses_low', 'avg_pts_for_low', 'avg_pts_against_low', 'avg_margin_low', 'avg_num_ot_low', 'last5_avg_pts_for_low', 'last5_avg_pts_against_low', 'last5_avg_margin_low', 'last5_win_rate_low', 'win_rate_low', 'games_high', 'wins_high', 'losses_high', 'avg_pts_for_high', 'avg_pts_against_high', 'avg_margin_high', 'avg_num_ot_high', 'last5_avg_pts_for_high', 'last5_avg_pts_against_high', 'last5_avg_margin_high', 'last5_win_rate_high', 'win_rate_high', 'consensus_low_spread', 'consensus_over_under', 'provider_count', 'diff_win_rate', 'diff_avg_margin', 'diff_avg_pts_for', 'diff_defense_proxy', 'diff_last5_win_rate', 'diff_last5_avg_margin']

Rows with all features populated: 0 / 132,133  (0.0%)


ID,season,team_low,team_high,sex,games_low,wins_low,losses_low,avg_pts_for_low,avg_pts_against_low,avg_margin_low,avg_num_ot_low,last5_avg_pts_for_low,last5_avg_pts_against_low,last5_avg_margin_low,last5_win_rate_low,win_rate_low,games_high,wins_high,losses_high,avg_pts_for_high,avg_pts_against_high,avg_margin_high,avg_num_ot_high,last5_avg_pts_for_high,last5_avg_pts_against_high,last5_avg_margin_high,last5_win_rate_high,win_rate_high,consensus_low_spread,consensus_over_under,provider_count,diff_win_rate,diff_avg_margin,diff_avg_pts_for,diff_defense_proxy,diff_last5_win_rate,diff_last5_avg_margin
str,i64,i64,i64,str,u32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64
"""2026_1101_1102""",2026,1101,1102,"""M""",30,11,19,67.933333,73.866667,-5.933333,0.033333,66.0,76.6,-10.6,0.2,0.366667,32,3,29,61.53125,79.875,-18.34375,0.0,61.2,78.4,-17.2,0.0,0.09375,null,null,0,0.272917,12.410417,6.402083,6.008333,0.2,6.6
"""2026_1101_1103""",2026,1101,1103,"""M""",30,11,19,67.933333,73.866667,-5.933333,0.033333,66.0,76.6,-10.6,0.2,0.366667,32,27,5,86.875,74.5,12.375,0.0,79.6,66.6,13.0,1.0,0.84375,null,null,0,-0.477083,-18.308333,-18.941667,0.633333,-0.8,-23.6
"""2026_1101_1104""",2026,1101,1104,"""M""",30,11,19,67.933333,73.866667,-5.933333,0.033333,66.0,76.6,-10.6,0.2,0.366667,32,23,9,91.71875,83.46875,8.25,0.0625,86.8,81.2,5.6,0.6,0.71875,null,null,0,-0.352083,-14.183333,-23.785417,9.602083,-0.4,-16.2


In [17]:
# --- Gold: game_features ---
# Aggregated features at the game level (used to build pairwise_features).
# avg_team_points / avg_opp_points are rolling season averages at the time of the game.
game_feats = pl.read_parquet(BASE / "gold/game_features.parquet")
print(f"game_features: {game_feats.height:,} rows")
print("Columns:", game_feats.columns)
game_feats.head(5)

game_features: 211,716 rows
Columns: ['sex', 'season', 'game_key', 'avg_team_points', 'avg_opp_points', 'sum_win_rows']


sex,season,game_key,avg_team_points,avg_opp_points,sum_win_rows
str,i64,str,f64,f64,i32
"""M""",2014,"""M_2014_33_1456_1459""",61.5,61.5,1
"""M""",2025,"""M_2025_87_1186_1226""",74.0,74.0,1
"""W""",2014,"""W_2014_128_3324_3464""",66.0,66.0,1
"""W""",2016,"""W_2016_14_3317_3328""",59.0,59.0,1
"""M""",2025,"""M_2025_22_1253_1431""",57.0,57.0,1


---
## Part 5 — CBBD Coverage & Mapping Quality

Two reports from the pipeline tell us how well the CBBD data integrates with the Kaggle data:
- **`cbbd_coverage_summary.json`** — how many games and teams were captured per season
- **`team_id_map_summary.json`** — what fraction of CBBD team names were successfully matched to Kaggle TeamIDs

In [18]:
# --- CBBD coverage by season ---
coverage = json.loads((BASE / "reports/cbbd_coverage_summary.json").read_text(encoding="utf-8"))
map_summary = json.loads((BASE / "reports/team_id_map_summary.json").read_text(encoding="utf-8"))

print("Team ID mapping summary:")
for k, v in map_summary.items():
    print(f"  {k}: {v}")

print()
coverage_df = pl.DataFrame(coverage["seasons"])
print("CBBD coverage per season:")
print(coverage_df)

Team ID mapping summary:
  total: 1209
  mapped: 366
  unmapped: 843
  mapped_pct: 0.3027295285359802

CBBD coverage per season:
shape: (10, 4)
┌────────┬────────────────┬───────┬──────────────┐
│ season ┆ team_game_rows ┆ games ┆ mapped_teams │
│ ---    ┆ ---            ┆ ---   ┆ ---          │
│ i64    ┆ i64            ┆ i64   ┆ i64          │
╞════════╪════════════════╪═══════╪══════════════╡
│ 2016   ┆ 11582          ┆ 5791  ┆ 349          │
│ 2017   ┆ 11560          ┆ 5781  ┆ 345          │
│ 2018   ┆ 11574          ┆ 5787  ┆ 347          │
│ 2019   ┆ 11738          ┆ 5869  ┆ 355          │
│ 2020   ┆ 11212          ┆ 5610  ┆ 350          │
│ 2021   ┆ 9930           ┆ 4965  ┆ 343          │
│ 2022   ┆ 11752          ┆ 5877  ┆ 355          │
│ 2023   ┆ 11614          ┆ 5810  ┆ 357          │
│ 2024   ┆ 11958          ┆ 5980  ┆ 357          │
│ 2025   ┆ 11786          ┆ 5894  ┆ 358          │
└────────┴────────────────┴───────┴──────────────┘


---
## Part 6 — Submission Output

The final output is `submission.csv` in the Kaggle-required format:
- `ID` = `Season_TeamLow_TeamHigh`  (team with the lower TeamID always comes first)
- `Pred` = probability that the team with the **lower** TeamID beats the team with the higher TeamID

Predictions use **dynamic clipping** based on seed matchup type:
- 1v16 matchups → `[0.005, 0.995]` (wider bounds for extreme mismatches)
- 2v15 matchups → `[0.01, 0.99]`
- 3v14 matchups → `[0.015, 0.985]`
- Other matchups → `[0.025, 0.975]` (default bounds)

Probabilities are also **isotonically calibrated** using out-of-fold predictions from time-series CV before clipping.

In [19]:
# --- Submission file ---
submission = pl.read_csv(BASE / "submission.csv")
print(f"Submission rows: {submission.height:,}")
print()
print("Prediction distribution (sanity check — should be centred near 0.5):")
print(submission["Pred"].describe())
print()
print("Sample rows:")
submission.head(10)

Submission rows: 132,133

Prediction distribution (sanity check — should be centred near 0.5):
shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ value    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 132133.0 │
│ null_count ┆ 0.0      │
│ mean       ┆ 0.485515 │
│ std        ┆ 0.291157 │
│ min        ┆ 0.025    │
│ 25%        ┆ 0.226374 │
│ 50%        ┆ 0.476719 │
│ 75%        ┆ 0.742345 │
│ max        ┆ 0.975    │
└────────────┴──────────┘

Sample rows:


ID,Pred
str,f64
"""2026_1101_1102""",0.867539
"""2026_1101_1103""",0.086674
"""2026_1101_1104""",0.133457
"""2026_1101_1105""",0.44235
"""2026_1101_1106""",0.530541
"""2026_1101_1107""",0.56261
"""2026_1101_1108""",0.781953
"""2026_1101_1110""",0.401717
"""2026_1101_1111""",0.23828


In [20]:
# --- Holdout validation: check model performance on 2025 tournament ---
# The pipeline holds out 2025 and evaluates log-loss on known tournament outcomes.
gold_summary = json.loads((BASE / "reports/gold_quality_summary.json").read_text(encoding="utf-8"))
print("Gold quality summary:")
for k, v in gold_summary.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Gold quality summary:
  total_rows: 132133
  good_rows: 132133
  bad_rows: 0


---
## Part 7 — End-to-End Trace: One Matchup

To make the pipeline concrete, let's trace a single team pairing all the way from raw data through to its final submission prediction.

In [21]:
# Pick the first submission row and trace it back through every layer
example_row = submission.head(1)
example_id = example_row["ID"][0]
season_str, t_low_str, t_high_str = example_id.split("_")
season = int(season_str)
t_low = int(t_low_str)
t_high = int(t_high_str)
pred = example_row["Pred"][0]

# Look up team names
def team_name(tid):
    row = team_dim.filter(pl.col("team_id") == tid)
    return row["team_name"][0] if row.height > 0 else str(tid)

name_low  = team_name(t_low)
name_high = team_name(t_high)

print(f"Matchup ID     : {example_id}")
print(f"Season         : {season}")
print(f"Team low  ({t_low}): {name_low}")
print(f"Team high ({t_high}): {name_high}")
print(f"Prediction     : {pred:.4f}  → {name_low} has a {pred:.1%} chance of winning")

Matchup ID     : 2026_1101_1102
Season         : 2026
Team low  (1101): Abilene Chr
Team high (1102): Air Force
Prediction     : 0.8675  → Abilene Chr has a 86.8% chance of winning


In [22]:
# Step 1 of trace — Silver: what were each team's season stats?
print(f"=== {name_low} ({t_low}) — Season {season} ===")
print(team_feats.filter((pl.col("team_id") == t_low) & (pl.col("season") == season)))

print(f"\n=== {name_high} ({t_high}) — Season {season} ===")
print(team_feats.filter((pl.col("team_id") == t_high) & (pl.col("season") == season)))

=== Abilene Chr (1101) — Season 2026 ===
shape: (1, 15)
┌─────┬────────┬─────────┬──────────────┬───┬──────────────┬──────────────┬─────────────┬──────────┐
│ sex ┆ season ┆ team_id ┆ games_played ┆ … ┆ last5_avg_pt ┆ last5_avg_ma ┆ last5_win_r ┆ win_rate │
│ --- ┆ ---    ┆ ---     ┆ ---          ┆   ┆ s_against    ┆ rgin         ┆ ate         ┆ ---      │
│ str ┆ i64    ┆ i64     ┆ u32          ┆   ┆ ---          ┆ ---          ┆ ---         ┆ f64      │
│     ┆        ┆         ┆              ┆   ┆ f64          ┆ f64          ┆ f64         ┆          │
╞═════╪════════╪═════════╪══════════════╪═══╪══════════════╪══════════════╪═════════════╪══════════╡
│ M   ┆ 2026   ┆ 1101    ┆ 30           ┆ … ┆ 76.6         ┆ -10.6        ┆ 0.2         ┆ 0.366667 │
└─────┴────────┴─────────┴──────────────┴───┴──────────────┴──────────────┴─────────────┴──────────┘

=== Air Force (1102) — Season 2026 ===
shape: (1, 15)
┌─────┬────────┬─────────┬──────────────┬───┬──────────────┬──────────────┬──────

In [23]:
# Step 2 of trace — Gold: what does the pairwise feature vector look like?
pair_row = pairs.filter(
    (pl.col("team_low") == t_low) & (pl.col("team_high") == t_high) & (pl.col("season") == season)
)
print(f"Pairwise feature row for {name_low} vs {name_high} ({season}):")
print(pair_row.to_pandas().T.to_string())

Pairwise feature row for Abilene Chr vs Air Force (2026):
                                         0
ID                          2026_1101_1102
season                                2026
team_low                              1101
team_high                             1102
sex                                      M
games_low                               30
wins_low                                11
losses_low                              19
avg_pts_for_low                  67.933333
avg_pts_against_low              73.866667
avg_margin_low                   -5.933333
avg_num_ot_low                    0.033333
last5_avg_pts_for_low                 66.0
last5_avg_pts_against_low             76.6
last5_avg_margin_low                 -10.6
last5_win_rate_low                     0.2
win_rate_low                      0.366667
games_high                              32
wins_high                                3
losses_high                             29
avg_pts_for_high                  61.53

In [24]:
# Step 3 of trace — Were there any betting lines for this team pair?
lines_for_pair = cbbd_consensus.filter(
    (pl.col("season") == season) &
    (
        ((pl.col("home_team_id").cast(pl.Int64) == t_low) | (pl.col("away_team_id").cast(pl.Int64) == t_low)) &
        ((pl.col("home_team_id").cast(pl.Int64) == t_high) | (pl.col("away_team_id").cast(pl.Int64) == t_high))
    )
)
if lines_for_pair.height > 0:
    print(f"Betting lines for {name_low} vs {name_high} in {season}:")
    print(lines_for_pair.select(["game_id", "start_date", "consensus_home_spread",
                                  "consensus_over_under", "provider_count"]))
else:
    print(f"No betting lines found for this specific matchup in {season}.")

No betting lines found for this specific matchup in 2026.


---
## Part 8 — Quick Reference: Column Glossary

| Column | Table | Meaning |
|---|---|---|
| `sex` | most | `'M'` = men's, `'W'` = women's |
| `season` | most | The year the season ended (e.g. 2026 = 2025–26 season) |
| `team_id` / `opp_team_id` | game_fact | Kaggle TeamID (men's 1000–1999, women's 3000–3999) |
| `team_low` / `team_high` | pairwise_features | Canonical matchup key: lower TeamID vs higher TeamID |
| `day_num` | game_fact | Days since Day 0 of that season (see MSeasons for calendar dates) |
| `win` | game_fact | 1 if this team won, 0 if they lost |
| `game_key` | game_fact, game_features | `season_teamlow_teamhigh` — stable cross-table join key |
| `row_quality_pass` | quality_flags | True if all required features are non-null for model training |
| `consensus_home_spread` | cbbd_line_consensus | Median spread across providers (negative = home favoured) |
| `provider_count` | cbbd_line_consensus | Number of providers that had a line for this game |
| `home_elo_start` / `away_elo_start` | bronze/cbbd/games | Elo rating entering the game (pre-game expectation) |
| `excitement` | bronze/cbbd/games | CBBD excitement index — higher = closer/more exciting game |
| `pace` | cbbd_game_fact | Estimated possessions per 40 minutes |
| `avg_margin` | team_season_features | Average point differential (pts_for − pts_against) per game |
| `sos` | team_season_features | Strength of schedule (mean ELO of opponents faced) |
| `fg_pct` / `fg3_pct` / `ft_pct` | team_season_features | Shooting efficiency: field goal %, 3-point %, free throw % |
| `opp_fg_pct` | team_season_features | Opponent field goal % (defensive quality) |
| `avg_reb_margin` | team_season_features | Average rebounding margin per game |
| `ast_to_ratio` | team_season_features | Assist-to-turnover ratio |
| `avg_stl_blk` | team_season_features | Average steals + blocks per game |
| `avg_possessions` | team_season_features | Tempo proxy (FGA − OREB + TO + 0.475×FTA) |
| `heat_delta` | heat_scores | actual_margin − expected_margin for a single game |
| `heat_1g` / `heat_3g` / `heat_5g` | heat_scores | Rolling heat averages over 1, 3, 5 games |
| `diff_*` | pairwise_features | team_low stat minus team_high stat (positive = team_low advantage) |
| `seed_product` | pairwise_features | seed_low × seed_high (interaction term) |
| `consensus_low_spread_filled` | pairwise_features | Betting spread for team_low (0.0 when unavailable) |
| `label` | pairwise_features | 1 if team_low won, 0 if team_high won (only present in training rows) |
| `Pred` | submission | P(team_low beats team_high), dynamically clipped by seed matchup |